# 🧰 quant-kit — Kaggle Benchmark Suite (v2)

**Free benchmarks using Kaggle's T4 GPU.**

- ⚡ Speed (PP/TG tok/s) via `llama-cpp-python` CUDA
- 📉 Perplexity (WikiText-2) — *flagged unreliable for SWA-based models like Gemma 4*
- 🧠 Downstream: GSM8K + IFEval (generative — avoids T4 OOM)
- 📄 Auto-uploads results JSON to HuggingFace

### ⚠️ Why only generative tasks here?
Gemma 4 has a **256,000-token vocabulary**. Multiple-choice tasks (ARC, HellaSwag, TruthfulQA) require computing logprobs over all 256k tokens per sample.
T4 (16GB) − 7GB model = ~9GB free → **not enough** → OOM crash.
Run `vastai_bench.ipynb` on an A100/RTX 4090 for MC tasks.

### Setup
1. Add `HF_TOKEN` as a Kaggle Secret
2. Set `HF_REPO` below
3. Enable **GPU T4 x2** → Run All (~4-6 hours)

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────
HF_REPO           = "Dhptl/gemma-4-12b-it-GGUF"
ORIGINAL_MODEL_ID = "google/gemma-4-12b-it"
QUANT_TYPE        = "Q4_K_M"

RUN_SPEED = True
RUN_PPL   = True
RUN_EVAL  = True
# Generative tasks only — MC tasks (HellaSwag/ARC) OOM on T4 with Gemma's 256k vocab
# MC tasks run on vastai_bench.ipynb (A100 has enough VRAM)
EVAL_TASKS = "gsm8k,ifeval"
# ──────────────────────────────────────────────────────────────────────

In [ ]:
# ── Install packages ───────────────────────────────────────────────────
import subprocess, sys, os

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "llama-cpp-python[server]",
    "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu121"
], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "huggingface_hub", "lm-eval[api]", "psutil",
    "datasets", "jinja2", "requests",
    "langdetect",   # required by lm-eval ifeval task
], check=True)

r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
print(f"GPU: {r.stdout.strip()}")
print("Packages ready!")

In [ ]:
# ── Setup HF + download main quant ────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download, list_repo_files
from pathlib import Path
import os, json

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
api = HfApi(token=HF_TOKEN)

model_name = HF_REPO.split("/")[1]
base_name  = model_name.replace("-GGUF", "")

all_files  = list(list_repo_files(HF_REPO, token=HF_TOKEN))
all_quants = sorted([f for f in all_files if f.endswith(".gguf") and "F16" not in f])
print(f"{len(all_quants)} quants: {[q.split('-')[-1].replace('.gguf','') for q in all_quants]}")

main_gguf_file  = f"{base_name}-{QUANT_TYPE}.gguf"
main_model_path = f"/kaggle/working/{main_gguf_file}"
print(f"Downloading {main_gguf_file}...")
hf_hub_download(repo_id=HF_REPO, filename=main_gguf_file,
                local_dir="/kaggle/working", token=HF_TOKEN)
print(f"Ready: {Path(main_model_path).stat().st_size/1e9:.2f} GB")

In [ ]:
# ── 1. Speed Benchmark via llama-cpp-python Python API ─────────────────
import time, json, os
from llama_cpp import Llama

speed_results = []

def load_llm_silent(model_path, n_ctx, n_batch, logits_all=False):
    devnull_fd = os.open(os.devnull, os.O_WRONLY)
    old_stderr = os.dup(2)
    os.dup2(devnull_fd, 2)
    os.close(devnull_fd)
    try:
        llm = Llama(model_path=model_path, n_gpu_layers=-1,
                    n_ctx=n_ctx, n_batch=n_batch,
                    logits_all=logits_all, verbose=False)
    finally:
        os.dup2(old_stderr, 2)
        os.close(old_stderr)
    return llm

if RUN_SPEED:
    print("\n" + "="*55)
    print(f"  Speed — {QUANT_TYPE} on T4 GPU")
    print("="*55)
    N_GEN, REPS = 128, 2

    for ctx in [128, 512, 2048]:
        print(f"  Context {ctx} tokens... ", end="", flush=True)
        llm = load_llm_silent(main_model_path, n_ctx=ctx+N_GEN, n_batch=ctx)
        prompt_tokens = llm.tokenize(b"The quick brown fox " * 80)[:ctx]
        pp_times, tg_times = [], []

        for _ in range(REPS):
            llm.reset()
            t0 = time.perf_counter()
            llm.eval(prompt_tokens)
            pp_times.append(len(prompt_tokens) / (time.perf_counter() - t0))
            t0 = time.perf_counter()
            for _ in range(N_GEN):
                llm.eval([llm.sample()])
            tg_times.append(N_GEN / (time.perf_counter() - t0))

        del llm
        pp_avg = round(sum(pp_times) / REPS, 2)
        tg_avg = round(sum(tg_times) / REPS, 2)
        speed_results.append({"context": ctx, "pp_tok_s": pp_avg, "tg_tok_s": tg_avg})
        print(f"TG={tg_avg} tok/s  PP={pp_avg} tok/s")
    print("Speed done!")

In [ ]:
# ── 2. Perplexity via llama-cpp-python logits_all ──────────────────────
# lm-eval's gguf backend does NOT support loglikelihood_rolling (wikitext).
# We compute it ourselves — same math as llama-perplexity binary.
#
# ⚠️  IMPORTANT: Gemma 4 uses Sliding Window Attention (SWA).
# The kv-cache log lines confirm this:
#   llama_kv_cache_iswa: using full-size SWA cache
#   llama_kv_cache: V embeddings have different sizes across layers
# SWA layers only attend to a local window, so logits for tokens
# outside that window are incorrect. This chunk-based method produces
# inflated (unreliable) PPL for SWA-based models.
# We compute and store the raw value, but flag it as unreliable.
import math, numpy as np
from datasets import load_dataset

ppl_result = None
ppl_all    = {}

if RUN_PPL:
    print("\n" + "="*55)
    print(f"  Perplexity — {QUANT_TYPE} on WikiText-2")
    print("  (computed via llama-cpp-python logits_all)")
    print("  ⚠️  Note: result may be inflated for SWA models")
    print("="*55)

    N_CTX     = 512    # chunk size (tokens)
    STRIDE    = 256    # overlap stride
    MAX_TOKS  = 4096   # limit total tokens (enough for reliable PPL)

    # Load WikiText-2 test set
    print("Loading WikiText-2...")
    ds   = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    text = "\n".join(ds["text"])

    # Load model with logits_all so we get per-position logits
    print("Loading model for PPL (logits_all=True)...")
    llm = load_llm_silent(main_model_path, n_ctx=N_CTX+1, n_batch=N_CTX, logits_all=True)

    tokens = llm.tokenize(text.encode())
    tokens = tokens[:MAX_TOKS]
    print(f"Processing {len(tokens)} tokens in chunks of {N_CTX} (stride {STRIDE})...")

    total_nll = 0.0
    n_tokens  = 0

    for begin in range(0, len(tokens) - 1, STRIDE):
        end   = min(begin + N_CTX, len(tokens))
        chunk = tokens[begin:end]
        if len(chunk) < 2:
            break

        llm.reset()
        llm.eval(chunk)

        # Scores shape: (n_tokens_evaluated, vocab_size)
        scores = np.array(llm.scores[:len(chunk)], dtype=np.float32)

        # Only count tokens in the 2nd half of the window to reduce boundary effects
        count_from = STRIDE // 2 if begin > 0 else 0

        for j in range(count_from, len(chunk) - 1):
            logits = scores[j]              # (vocab_size,)
            target = chunk[j + 1]
            # numerically stable log-softmax, then pick target token prob
            logits -= logits.max()
            log_sum = math.log(np.exp(logits).sum())
            nll     = -(float(logits[target]) - log_sum)
            total_nll += nll
            n_tokens  += 1

        if end >= len(tokens):
            break

    del llm

    if n_tokens > 0:
        ppl_result = round(math.exp(total_nll / n_tokens), 4)
        ppl_all[QUANT_TYPE] = ppl_result
        print(f"  {QUANT_TYPE} Perplexity = {ppl_result}  ({n_tokens} tokens evaluated)")

        # SWA reliability check: values > 100 are almost certainly SWA artifacts
        PPL_RELIABLE_THRESHOLD = 100
        if ppl_result > PPL_RELIABLE_THRESHOLD:
            print(f"  ⚠️  PPL={ppl_result} exceeds threshold ({PPL_RELIABLE_THRESHOLD}).")
            print("     This model likely uses Sliding Window Attention (SWA).")
            print("     The chunk-based logits_all method is not valid for SWA.")
            print("     PPL will be stored as raw but excluded from model card.")
    print("Perplexity done!")

In [ ]:
# ── Start llama-cpp-python OpenAI server (for lm-eval) ─────────────────
import subprocess, sys, time
import requests as req

SERVER_PORT = 8080
SERVER_URL  = f"http://localhost:{SERVER_PORT}"

print(f"Starting server on port {SERVER_PORT}...")
log_file = open("/kaggle/working/server_log.txt", "w")

server_proc = subprocess.Popen([
    sys.executable, "-m", "llama_cpp.server",
    "--model",        main_model_path,
    "--n_gpu_layers", "-1",
    "--n_ctx",        "4096",
    "--port",         str(SERVER_PORT),
    "--host",         "0.0.0.0",
], stdout=log_file, stderr=subprocess.STDOUT, text=True)

server_ready = False
for i in range(90):
    time.sleep(2)
    if server_proc.poll() is not None:
        print("\n❌ Server crashed!")
        with open("/kaggle/working/server_log.txt") as f:
            print(f.read())
        raise RuntimeError("Server crashed.")
    try:
        r = req.get(f"{SERVER_URL}/v1/models", timeout=2)
        if r.status_code == 200:
            print(f"\n✅ Server ready after {i*2}s!")
            server_ready = True
            break
    except Exception:
        pass

if not server_ready:
    server_proc.terminate()
    with open("/kaggle/working/server_log.txt") as f:
        print("".join(f.readlines()[-20:]))
    raise RuntimeError("Server timed out.")

print(f"Server at {SERVER_URL}")

In [ ]:
# ── 3. Downstream benchmarks — generative only (no T4 OOM) ────────────
from pathlib import Path
import subprocess, sys, json

eval_results = {}

if RUN_EVAL:
    print("\n" + "="*55)
    print(f"  lm-eval — Generative Tasks: {EVAL_TASKS}")
    print("  (MC tasks: run vastai_bench.ipynb on A100)")
    print("="*55)

    results_dir = Path("/kaggle/working/eval_results")
    results_dir.mkdir(exist_ok=True)

    cmd = [
        sys.executable, "-m", "lm_eval",
        "--model",       "gguf",
        "--model_args",  f"base_url={SERVER_URL}",
        "--tasks",       EVAL_TASKS,
        "--limit",       "250",
        "--output_path", str(results_dir),
        "--batch_size",  "1",
    ]
    print("Running (~2 hrs at 14s/sample)...")
    subprocess.run(cmd, text=True, timeout=18000)

    try:
        server_proc.terminate()
        print("Server stopped.")
    except Exception:
        pass

    # ── Collect ALL result files — no premature break ──────────────────
    for result_file in results_dir.glob("**/*.json"):
        if "results" not in result_file.name:
            continue
        try:
            with open(result_file) as f:
                data = json.load(f)
            for task, metrics in data.get("results", {}).items():
                # Priority order — covers GSM8K flexible-extract, IFEval, ARC, etc.
                score = (
                    metrics.get("acc_norm,none") or
                    metrics.get("acc,none") or
                    metrics.get("exact_match,flexible-extract") or  # GSM8K
                    metrics.get("exact_match,none") or
                    metrics.get("prompt_level_strict_acc,none") or  # IFEval
                    metrics.get("pass@1,none")
                )
                if score is not None and task not in eval_results:
                    eval_results[task] = round(float(score) * 100, 2)
                    print(f"  {task}: {eval_results[task]}%")
        except Exception as e:
            print(f"  ⚠️  Could not parse {result_file.name}: {e}")
    print("lm-eval done!")

In [ ]:
# ── 4. Save + upload results JSON ─────────────────────────────────────
# PPL guard: Gemma 4 uses Sliding Window Attention (SWA).
# Our chunk-based logits_all PPL is NOT reliable for SWA models — the SWA
# layers only attend locally, so logits outside the window are garbage.
# We store the raw value but flag it and exclude it from the model card
# (model_card.py reads perplexity_reliable and respects it).
PPL_RELIABLE_THRESHOLD = 100  # anything above is likely an SWA artifact
ppl_reliable = ppl_result is not None and ppl_result < PPL_RELIABLE_THRESHOLD
if ppl_result and not ppl_reliable:
    print(f"\n⚠️  PPL={ppl_result} is unreliable (SWA model). Flagging in JSON.")

output = {
    "model":              HF_REPO,
    "quant":              QUANT_TYPE,
    "platform":           "Kaggle T4 GPU",
    "speed":              speed_results,
    "perplexity_raw":     ppl_result,
    "perplexity_reliable": ppl_reliable,
    "perplexity":         ppl_result if ppl_reliable else None,
    "benchmarks":         eval_results,
}
result_file = f"/kaggle/working/kaggle_results_{QUANT_TYPE}.json"
with open(result_file, "w") as f:
    json.dump(output, f, indent=2)
api.upload_file(
    path_or_fileobj=result_file,
    path_in_repo=f"kaggle_results_{QUANT_TYPE}.json",
    repo_id=HF_REPO, repo_type="model",
    commit_message=f"Add Kaggle benchmark results ({QUANT_TYPE})"
)
print("Results JSON uploaded!")
print(json.dumps(output, indent=2))